# Wikipedia Pets Content Feature Analysis

This notebook computes one row of text-level content features for every article in the cleaned Wikipedia pets corpus.

The workflow is CPU-only, never downloads models automatically, and writes generated artifacts under `data/content_features/`.

## 1. Project Overview And Goals

The feature table covers document identity, surface statistics, readability, lexical complexity, lightweight sentiment, and approximate entity extraction.

Token counts reuse the hybrid retriever's BM25 tokenizer directly. Sentence splitting mirrors the MiniLM embedding workflow's lightweight `spacy.blank('en')` plus `sentencizer` setup.

## 2. Imports And Paths

Required packages are `pandas`, `numpy`, and `spacy`. Optional packages are detected gracefully.

In [13]:
from collections import Counter
from datetime import datetime, timezone
from importlib.util import find_spec
from pathlib import Path
import json
import re
import sys
import time
import warnings

import numpy as np
import pandas as pd
import spacy

try:
    from tqdm.auto import tqdm
    TQDM_AVAILABLE = True
except ImportError:
    TQDM_AVAILABLE = False

    def tqdm(iterable, **_kwargs):
        return iterable

PYARROW_AVAILABLE = find_spec('pyarrow') is not None

print(f'tqdm available: {TQDM_AVAILABLE}')
print(f'pyarrow available: {PYARROW_AVAILABLE}')

tqdm available: True
pyarrow available: True


In [14]:
REPO_ROOT_CANDIDATES = [
    Path.cwd().resolve(),
    Path.cwd().resolve().parent,
    Path.cwd().resolve().parent.parent,
]
REPO_ROOT = next(
    (
        path
        for path in REPO_ROOT_CANDIDATES
        if (path / 'data').exists() and (path / 'content_features_analysis').exists()
    ),
    Path.cwd().resolve(),
)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

DEFAULT_INPUT_DIR = REPO_ROOT / 'data' / 'corpus_cleaned'
DEFAULT_OUTPUT_DIR = REPO_ROOT / 'data' / 'content_features'

FEATURES_PARQUET_FILE = DEFAULT_OUTPUT_DIR / 'content_features.parquet'
FEATURES_CSV_FILE = DEFAULT_OUTPUT_DIR / 'content_features.csv'
CONFIG_FILE = DEFAULT_OUTPUT_DIR / 'content_features_config.json'
print(f'Repo root: {REPO_ROOT}')
print(f'Default input directory: {DEFAULT_INPUT_DIR}')
print(f'Default output directory: {DEFAULT_OUTPUT_DIR}')

Repo root: D:\University\FCSE\Courses\26S\TUG-26S-BT-Content_Optimization
Default input directory: D:\University\FCSE\Courses\26S\TUG-26S-BT-Content_Optimization\data\corpus_cleaned
Default output directory: D:\University\FCSE\Courses\26S\TUG-26S-BT-Content_Optimization\data\content_features


## 3. Configuration

Set `DOCUMENT_LIMIT` to a small integer for a quick test run.

In [15]:
INPUT_DIR = DEFAULT_INPUT_DIR
OUTPUT_DIR = DEFAULT_OUTPUT_DIR

DOCUMENT_LIMIT = None
SAMPLE_DOCUMENT_COUNT = 3
PROGRESS_EVERY = 500

TOP_ENTITY_LIMIT = 20
ENTITY_MIN_FREQUENCY = 2
SPACY_NER_MODEL = 'en_core_web_sm'
NER_BATCH_SIZE = 16

FEATURES_PARQUET_FILE = OUTPUT_DIR / 'content_features.parquet'
FEATURES_CSV_FILE = OUTPUT_DIR / 'content_features.csv'
CONFIG_FILE = OUTPUT_DIR / 'content_features_config.json'

print(f'Input directory: {INPUT_DIR}')
print(f'Output directory: {OUTPUT_DIR}')
print(f'Document limit: {DOCUMENT_LIMIT}')

Input directory: D:\University\FCSE\Courses\26S\TUG-26S-BT-Content_Optimization\data\corpus_cleaned
Output directory: D:\University\FCSE\Courses\26S\TUG-26S-BT-Content_Optimization\data\content_features
Document limit: None


## 4. Loading Corpus Files

The corpus is discovered recursively, sorted deterministically, and read with UTF-8 replacement handling so malformed text does not stop the run.

In [16]:
READ_ERRORS = []


def relative_to_repo(path):
    try:
        return path.resolve().relative_to(REPO_ROOT).as_posix()
    except ValueError:
        return str(path.resolve())


def discover_corpus_files(input_dir, document_limit=None):
    if not input_dir.exists():
        raise FileNotFoundError(f'Input directory does not exist: {input_dir}')

    files = sorted(
        input_dir.rglob('*.txt'),
        key=lambda path: path.relative_to(input_dir).as_posix().lower(),
    )
    if document_limit is not None:
        files = files[: max(int(document_limit), 0)]
    return files


def read_text_file(path, error_log=None):
    try:
        return path.read_text(encoding='utf-8', errors='replace')
    except Exception as exc:
        if error_log is not None:
            error_log.append({'source_path': relative_to_repo(path), 'error': str(exc)})
        return ''


corpus_files = discover_corpus_files(INPUT_DIR, DOCUMENT_LIMIT)
if not corpus_files:
    raise FileNotFoundError(f'No .txt files were found under {INPUT_DIR}')

print(f'Discovered {len(corpus_files):,} corpus files.')
print(f'First file: {relative_to_repo(corpus_files[0])}')

Discovered 21,058 corpus files.
First file: data/corpus_cleaned/(Blooper)_Bunny_8dbe4cbf9c.txt


## 5. Feature Extraction Helpers

`token_count` uses `retriever.run_hybrid_rankings.tokenize_for_bm25` directly, so it follows the same normalization, regex, stopword removal, number handling, and minimum token length as the retriever. Readability metrics use the same normalized alphabetic regex stream before BM25 filtering because readability formulas need common words such as stopwords.

In [17]:
from retriever.run_hybrid_rankings import (
    STOPWORDS as RETRIEVER_STOPWORDS,
    normalize_text as normalize_retriever_text,
    tokenize_for_bm25,
)

LANGUAGE_WORD_RE = re.compile(r"[a-z]+(?:['’][a-z]+)*")
HASH_SUFFIX_RE = re.compile(r'_[0-9a-f]{10}$', flags=re.IGNORECASE)
VOWELS = set('aeiouy')
STOPWORDS = set(RETRIEVER_STOPWORDS)


def safe_divide(numerator, denominator, default=0.0):
    return float(numerator) / float(denominator) if denominator else default


def tokenize_text(text):
    return tokenize_for_bm25(text)


def tokenize_words_for_language_metrics(text):
    normalized = normalize_retriever_text(text)
    raw_words = LANGUAGE_WORD_RE.findall(normalized)
    return [normalize_word(word) for word in raw_words if normalize_word(word)]


def normalize_word(token):
    return str(token).replace("'", '').replace('’', '').strip().lower()


def title_from_path(path):
    stem = HASH_SUFFIX_RE.sub('', path.stem)
    title = stem.replace('_', ' ')
    return re.sub(r'\s+', ' ', title).strip() or path.stem


def count_syllables(word):
    normalized = re.sub(r'[^a-z]', '', str(word).lower())
    if not normalized:
        return 0
    if len(normalized) <= 3:
        return 1

    syllables = 0
    previous_was_vowel = False
    for character in normalized:
        is_vowel = character in VOWELS
        if is_vowel and not previous_was_vowel:
            syllables += 1
        previous_was_vowel = is_vowel

    if normalized.endswith('e') and not normalized.endswith(('le', 'ye')) and syllables > 1:
        syllables -= 1
    return max(1, syllables)


def lexical_metrics(words):
    word_count = len(words)
    lexical_words = [word for word in words if word.isalpha() and word not in STOPWORDS]
    unique_words = set(words)
    return {
        'lexical_density': safe_divide(len(lexical_words), word_count),
        'lexical_diversity': safe_divide(len(unique_words), word_count),
    }

## 6. Sentence Splitting Utilities

Sentence splitting mirrors the MiniLM embedding notebook: a separate lightweight blank English pipeline with a sentencizer and the same maximum-length floor. The NER model is not used for sentence boundaries.

In [18]:
SENTENCE_NLP = spacy.blank('en')
if 'sentencizer' not in SENTENCE_NLP.pipe_names:
    SENTENCE_NLP.add_pipe('sentencizer')
SENTENCE_NLP.max_length = max(SENTENCE_NLP.max_length, 5_000_000)


def split_sentences(text):
    cleaned = str(text or '').strip()
    if not cleaned:
        return []
    if len(cleaned) >= SENTENCE_NLP.max_length:
        SENTENCE_NLP.max_length = len(cleaned) + 1_000
    document = SENTENCE_NLP(cleaned)
    return [sentence.text.strip() for sentence in document.sents if sentence.text.strip()]

## 7. Readability Metrics

Flesch-Kincaid Grade Level and Gunning Fog Index use the formulas specified for this project. Empty documents return null readability scores.

In [19]:
def flesch_kincaid_grade(word_count, sentence_count, syllable_count):
    if word_count <= 0 or sentence_count <= 0:
        return np.nan
    return 0.39 * (word_count / sentence_count) + 11.8 * (syllable_count / word_count) - 15.59


def gunning_fog_index(word_count, sentence_count, complex_word_count):
    if word_count <= 0 or sentence_count <= 0:
        return np.nan
    return 0.4 * ((word_count / sentence_count) + 100 * (complex_word_count / word_count))


def readability_metrics(words, sentence_count, entity_words=None):
    entity_words = entity_words or set()
    syllables_per_word = [count_syllables(word) for word in words]
    syllable_count = int(sum(syllables_per_word))
    complex_word_count = sum(
        1
        for word, syllables in zip(words, syllables_per_word)
        if len(word) >= 4 and syllables >= 3 and word not in entity_words
    )
    word_count = len(words)
    return {
        'syllable_count': syllable_count,
        'avg_syllables_per_word': safe_divide(syllable_count, word_count),
        'flesch_kincaid_grade': flesch_kincaid_grade(word_count, sentence_count, syllable_count),
        'gunning_fog_index': gunning_fog_index(word_count, sentence_count, complex_word_count),
    }

## 8. Sentiment Analysis

VADER is used when `vaderSentiment` is installed. Its compound score becomes `sentiment_polarity`. VADER does not expose subjectivity, so `sentiment_subjectivity` remains null.

In [20]:
try:
    from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

    SENTIMENT_ANALYZER = SentimentIntensityAnalyzer()
    VADER_AVAILABLE = True
    print('VADER sentiment is available.')
except Exception as exc:
    SENTIMENT_ANALYZER = None
    VADER_AVAILABLE = False
    print(f'VADER sentiment is unavailable; sentiment values will be null. Reason: {exc}')


def sentiment_metrics(text):
    if SENTIMENT_ANALYZER is None or not str(text or '').strip():
        return {'sentiment_polarity': np.nan, 'sentiment_subjectivity': np.nan}
    scores = SENTIMENT_ANALYZER.polarity_scores(text)
    return {
        'sentiment_polarity': float(scores.get('compound', np.nan)),
        'sentiment_subjectivity': np.nan,
    }

VADER sentiment is available.


## 9. Entity Extraction

General named entities come from the local spaCy `en_core_web_sm` NER pipeline. Regex enrichment is retained for structured values that statistical NER often misses, such as email addresses and URLs. The notebook never downloads a model automatically.

In [21]:
STRUCTURED_ENTITY_PATTERNS = {
    'EMAIL': re.compile(r'\b[A-Z0-9._%+-]+@[A-Z0-9.-]+\.[A-Z]{2,}\b', flags=re.IGNORECASE),
    'URL': re.compile(r'\b(?:https?://|www\.)[^\s<>{}\[\]]+', flags=re.IGNORECASE),
}

try:
    ENTITY_NLP = spacy.load(
        SPACY_NER_MODEL,
        disable=['tagger', 'parser', 'attribute_ruler', 'lemmatizer'],
    )
except OSError as exc:
    raise RuntimeError(
        f'Missing spaCy NER model {SPACY_NER_MODEL!r}. Install it manually before running this notebook; '
        'the notebook will not download models automatically.'
    ) from exc

ENTITY_NLP.max_length = max(ENTITY_NLP.max_length, 5_000_000)
print(f'Loaded spaCy NER model: {SPACY_NER_MODEL} with pipes {ENTITY_NLP.pipe_names}')


def normalize_entity_text(text):
    return re.sub(r'\s+', ' ', str(text)).strip(' \t\r\n.,;:!?()[]{}')


def spans_overlap(left, right):
    return left[0] < right[1] and right[0] < left[1]


def extract_entities(text, entity_doc=None):
    cleaned_text = str(text or '')
    if not cleaned_text.strip():
        return {
            'entity_count': 0,
            'entities_json': '[]',
            'top_entities_json': '[]',
            'entity_words': set(),
        }

    if len(cleaned_text) >= ENTITY_NLP.max_length:
        ENTITY_NLP.max_length = len(cleaned_text) + 1_000
    if entity_doc is None:
        entity_doc = ENTITY_NLP(cleaned_text)

    counts = Counter()
    structured_spans = []
    for label, pattern in STRUCTURED_ENTITY_PATTERNS.items():
        for match in pattern.finditer(cleaned_text):
            entity_text = normalize_entity_text(match.group(0))
            if entity_text:
                counts[(entity_text, label)] += 1
                structured_spans.append((match.start(), match.end()))

    for entity in entity_doc.ents:
        entity_span = (entity.start_char, entity.end_char)
        if any(spans_overlap(entity_span, span) for span in structured_spans):
            continue
        entity_text = normalize_entity_text(entity.text)
        if entity_text:
            counts[(entity_text, entity.label_)] += 1

    entity_records = [
        {'entity': entity, 'label': label, 'count': int(count)}
        for (entity, label), count in sorted(
            counts.items(),
            key=lambda item: (-item[1], item[0][0].lower(), item[0][1]),
        )
    ]
    top_entity_records = [
        record for record in entity_records if record['count'] >= ENTITY_MIN_FREQUENCY
    ][:TOP_ENTITY_LIMIT]
    entity_words = {
        word
        for record in entity_records
        for word in tokenize_words_for_language_metrics(record['entity'])
        if word.isalpha()
    }
    return {
        'entity_count': int(sum(counts.values())),
        'entities_json': json.dumps(entity_records, ensure_ascii=False),
        'top_entities_json': json.dumps(top_entity_records, ensure_ascii=False),
        'entity_words': entity_words,
    }

Loaded spaCy NER model: en_core_web_sm with pipes ['tok2vec', 'ner']


## 10. Processing Sample Documents

The sample pass is a quick sanity check before processing the full corpus.

In [22]:
def extract_document_features(path, doc_id, text=None, entity_doc=None, error_log=None):
    if text is None:
        text = read_text_file(path, error_log=error_log)
    sentences = split_sentences(text)
    tokens = tokenize_text(text)
    words = tokenize_words_for_language_metrics(text)

    entity_values = extract_entities(text, entity_doc=entity_doc)
    lexical_values = lexical_metrics(words)
    readability_values = readability_metrics(words, len(sentences), entity_values['entity_words'])
    sentiment_values = sentiment_metrics(text)

    token_lengths = [len(token) for token in tokens if token]
    row = {
        'doc_id': int(doc_id),
        'title': title_from_path(path),
        'source_path': relative_to_repo(path),
        'char_count': len(text),
        'non_whitespace_char_count': sum(1 for character in text if not character.isspace()),
        'word_count': len(words),
        'token_count': len(tokens),
        'sentence_count': len(sentences),
        'avg_words_per_sentence': safe_divide(len(words), len(sentences)),
        'avg_token_length': safe_divide(sum(token_lengths), len(token_lengths)),
        **readability_values,
        **lexical_values,
        **sentiment_values,
        'entity_count': entity_values['entity_count'],
        'entities_json': entity_values['entities_json'],
        'top_entities_json': entity_values['top_entities_json'],
    }
    return row


sample_paths = corpus_files[: min(SAMPLE_DOCUMENT_COUNT, len(corpus_files))]
sample_features_df = pd.DataFrame(
    [extract_document_features(path, doc_id) for doc_id, path in enumerate(sample_paths)]
)
sample_features_df[['doc_id', 'title', 'word_count', 'sentence_count', 'flesch_kincaid_grade', 'entity_count']]

,doc_id,title,word_count,sentence_count,flesch_kincaid_grade,entity_count
0,0,(Blooper) Bunny,1737,69,12.760008,175
1,1,0.0. Duck,19637,915,11.269760,2869
2,2,100th Anniversary of the Canadian Navy,465,19,14.611081,51


## 11. Processing The Full Corpus

This is the main corpus pass. Progress is reported through `tqdm` when available and through periodic log messages in all environments.

In [23]:
FEATURE_COLUMNS = [
    'doc_id',
    'title',
    'source_path',
    'char_count',
    'non_whitespace_char_count',
    'word_count',
    'token_count',
    'sentence_count',
    'avg_words_per_sentence',
    'avg_token_length',
    'syllable_count',
    'avg_syllables_per_word',
    'flesch_kincaid_grade',
    'gunning_fog_index',
    'lexical_density',
    'lexical_diversity',
    'sentiment_polarity',
    'sentiment_subjectivity',
    'entity_count',
    'entities_json',
    'top_entities_json',
]


def iter_ner_inputs(paths):
    for doc_id, path in enumerate(paths):
        text = read_text_file(path, error_log=READ_ERRORS)
        if len(text) >= ENTITY_NLP.max_length:
            ENTITY_NLP.max_length = len(text) + 1_000
        yield text, (doc_id, path)


def process_corpus(paths):
    rows = []
    start_time = time.time()
    ner_docs = ENTITY_NLP.pipe(iter_ner_inputs(paths), as_tuples=True, batch_size=NER_BATCH_SIZE)
    progress = tqdm(ner_docs, total=len(paths), desc='Extracting content features')
    for entity_doc, (doc_id, path) in progress:
        rows.append(
            extract_document_features(
                path,
                doc_id,
                text=entity_doc.text,
                entity_doc=entity_doc,
                error_log=READ_ERRORS,
            )
        )
        processed_count = doc_id + 1
        if processed_count % PROGRESS_EVERY == 0 or processed_count == len(paths):
            elapsed = time.time() - start_time
            print(f'Processed {processed_count:,}/{len(paths):,} documents in {elapsed:.1f}s')
    return pd.DataFrame(rows).reindex(columns=FEATURE_COLUMNS)


READ_ERRORS.clear()
features_df = process_corpus(corpus_files)

print(f'Feature rows: {len(features_df):,}')
print(f'Read errors: {len(READ_ERRORS):,}')
features_df.head()

Extracting content features:   2%|▏         | 501/21058 [02:09<46:26,  7.38it/s]  

Processed 500/21,058 documents in 129.5s


Extracting content features:   5%|▍         | 1006/21058 [03:42<46:20,  7.21it/s] 

Processed 1,000/21,058 documents in 222.2s


Extracting content features:   7%|▋         | 1501/21058 [09:44<1:24:45,  3.85it/s] 

...

Extracting content features: 100%|█████████▉| 21003/21058 [3:03:32<00:06,  9.13it/s]

Processed 21,000/21,058 documents in 11012.9s


Extracting content features: 100%|██████████| 21058/21058 [3:03:37<00:00,  1.91it/s]


Processed 21,058/21,058 documents in 11017.8s
Feature rows: 21,058
Read errors: 0


,doc_id,title,source_path,char_count,non_whitespace_char_count,word_count,token_count,sentence_count,avg_words_per_sentence,avg_token_length,...,avg_syllables_per_word,flesch_kincaid_grade,gunning_fog_index,lexical_density,lexical_diversity,sentiment_polarity,sentiment_subjectivity,entity_count,entities_json,top_entities_json
0,0,(Blooper) Bunny,data/corpus_cleaned/(Blooper)_Bunny_8dbe4cbf9c...,10425,8620,1737,1030,69,25.173913,6.033981,...,1.570524,12.760008,14.928517,0.583189,0.397237,0.8722,NaN,175,"[{""entity"": ""Daffy"", ""label"": ""ORG"", ""count"": ...","[{""entity"": ""Daffy"", ""label"": ""ORG"", ""count"": ..."
1,1,0.0. Duck,data/corpus_cleaned/0.0._Duck_c54e2adeaa.txt,117303,96828,19637,11596,915,21.461202,6.079251,...,1.566940,11.269760,12.475096,0.583185,0.191679,1.0000,NaN,2869,"[{""entity"": ""Scrooge"", ""label"": ""PERSON"", ""cou...","[{""entity"": ""Scrooge"", ""label"": ""PERSON"", ""cou..."
2,2,100th Anniversary of the Canadian Navy,data/corpus_cleaned/100th_Anniversary_of_the_C...,2936,2459,465,284,19,24.473684,6.683099,...,1.750538,14.611081,15.810979,0.593548,0.466667,0.9747,NaN,51,"[{""entity"": ""Canadian"", ""label"": ""NORP"", ""coun...","[{""entity"": ""Canadian"", ""label"": ""NORP"", ""coun..."
3,3,101 Dalmatians (1996 film),data/corpus_cleaned/101_Dalmatians_(1996_film)...,8564,7076,1400,881,81,17.283951,5.903519,...,1.613571,10.190884,10.970723,0.595714,0.409286,-0.9118,NaN,231,"[{""entity"": ""Anita"", ""label"": ""PERSON"", ""count...","[{""entity"": ""Anita"", ""label"": ""PERSON"", ""count..."
4,4,101 Shark Pets,data/corpus_cleaned/101_Shark_Pets_a28f849c9b.txt,1189,959,216,126,9,24.000000,5.571429,...,1.388889,10.158889,12.562963,0.564815,0.532407,0.8402,NaN,13,"[{""entity"": ""101"", ""label"": ""CARDINAL"", ""count...","[{""entity"": ""101"", ""label"": ""CARDINAL"", ""count..."


## 12. Saving Outputs

CSV is always written. Parquet is written when `pyarrow` is available.

In [24]:
def write_parquet_with_warning(dataframe, path, label):
    if not PYARROW_AVAILABLE:
        warnings.warn(f'pyarrow is unavailable; skipping {label} parquet output: {path}')
        return False
    try:
        dataframe.to_parquet(path, index=False)
        return True
    except Exception as exc:
        warnings.warn(f'Could not write {label} parquet output {path}: {exc}')
        return False


OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
features_df.to_csv(FEATURES_CSV_FILE, index=False, encoding='utf-8')
features_parquet_written = write_parquet_with_warning(features_df, FEATURES_PARQUET_FILE, 'content features')

run_config = {
    'generated_at_utc': datetime.now(timezone.utc).isoformat(),
    'input_directory': relative_to_repo(INPUT_DIR),
    'output_directory': relative_to_repo(OUTPUT_DIR),
    'document_limit': DOCUMENT_LIMIT,
    'input_file_count': len(corpus_files),
    'processed_document_count': len(features_df),
    'read_error_count': len(READ_ERRORS),
    'read_errors': READ_ERRORS,
    'package_availability': {
        'tqdm': TQDM_AVAILABLE,
        'pyarrow': PYARROW_AVAILABLE,
        'vaderSentiment': VADER_AVAILABLE,
        'spacy_ner_model': SPACY_NER_MODEL,
    },
    'parameters': {
        'top_entity_limit': TOP_ENTITY_LIMIT,
        'entity_min_frequency': ENTITY_MIN_FREQUENCY,
        'ner_batch_size': NER_BATCH_SIZE,
        'tokenizer': 'retriever.run_hybrid_rankings.tokenize_for_bm25',
        'sentence_splitter': "spacy.blank('en') + sentencizer",
    },
    'formulas': {
        'flesch_kincaid_grade': '0.39 * (words / sentences) + 11.8 * (syllables / words) - 15.59',
        'gunning_fog_index': '0.4 * ((words / sentences) + 100 * (complex_words / words))',
        'lexical_density': 'non_stopword_alphabetic_words / word_count',
        'lexical_diversity': 'unique_lowercase_words / word_count',
    },
}
CONFIG_FILE.write_text(json.dumps(run_config, indent=2), encoding='utf-8')

print(f'CSV written: {FEATURES_CSV_FILE}')
print(f'Parquet written: {FEATURES_PARQUET_FILE}' if features_parquet_written else 'Parquet output was skipped.')
print(f'Config written: {CONFIG_FILE}')

CSV written: D:\University\FCSE\Courses\26S\TUG-26S-BT-Content_Optimization\data\content_features\content_features.csv
Parquet written: D:\University\FCSE\Courses\26S\TUG-26S-BT-Content_Optimization\data\content_features\content_features.parquet
Config written: D:\University\FCSE\Courses\26S\TUG-26S-BT-Content_Optimization\data\content_features\content_features_config.json


## 13. Basic Descriptive Analysis

This summary provides a quick check for unexpected values and corpus-wide scale.

In [25]:
SUMMARY_COLUMNS = [
    'char_count',
    'word_count',
    'sentence_count',
    'avg_words_per_sentence',
    'avg_token_length',
    'avg_syllables_per_word',
    'flesch_kincaid_grade',
    'gunning_fog_index',
    'lexical_density',
    'lexical_diversity',
    'sentiment_polarity',
    'entity_count',
]

descriptive_summary = features_df[SUMMARY_COLUMNS].describe().T
descriptive_summary

,count,mean,std,min,25%,50%,75%,max
char_count,21058.0,7487.345237,18652.035854,39.000000,507.000000,1819.000000,4733.750000,279675.000000
word_count,21058.0,1182.738152,2902.713014,7.000000,79.000000,294.000000,763.000000,47645.000000
sentence_count,21058.0,57.892013,144.464155,1.000000,5.000000,15.000000,37.000000,2683.000000
avg_words_per_sentence,21058.0,19.239603,5.595925,3.271565,16.000000,19.340944,22.515364,251.750000
avg_token_length,21058.0,6.279056,0.545396,4.354167,5.899683,6.212947,6.599473,10.000000
avg_syllables_per_word,21058.0,1.692436,0.169245,1.214286,1.578202,1.657143,1.775510,2.907407
flesch_kincaid_grade,21058.0,11.884195,2.701662,1.468571,10.154091,11.639631,13.196017,101.560971
gunning_fog_index,21058.0,12.719058,2.732968,2.800000,10.965818,12.632312,14.310416,104.692056
lexical_density,21058.0,0.607355,0.059866,0.424242,0.571578,0.600000,0.629786,0.950111
lexical_diversity,21058.0,0.545916,0.168938,0.076147,0.433405,0.534308,0.675863,1.000000


## 14. Numeric Correlation Matrix

Correlations are descriptive only. They do not imply causal relationships.

In [26]:
CORRELATION_COLUMNS = [
    'char_count',
    'word_count',
    'sentence_count',
    'avg_words_per_sentence',
    'avg_token_length',
    'avg_syllables_per_word',
    'flesch_kincaid_grade',
    'gunning_fog_index',
    'lexical_density',
    'lexical_diversity',
    'sentiment_polarity',
    'entity_count',
]

correlation_matrix = features_df[CORRELATION_COLUMNS].corr().round(3)
correlation_matrix

,char_count,word_count,sentence_count,avg_words_per_sentence,avg_token_length,avg_syllables_per_word,flesch_kincaid_grade,gunning_fog_index,lexical_density,lexical_diversity,sentiment_polarity,entity_count
char_count,1.000,0.998,0.983,0.138,-0.115,-0.065,0.064,-0.024,0.061,-0.586,-0.181,0.969
word_count,0.998,1.000,0.978,0.144,-0.122,-0.080,0.057,-0.021,0.049,-0.593,-0.174,0.957
sentence_count,0.983,0.978,1.000,0.085,-0.131,-0.071,0.016,-0.074,0.059,-0.578,-0.184,0.960
avg_words_per_sentence,0.138,0.144,0.085,1.000,-0.128,-0.167,0.685,0.728,0.087,-0.412,0.141,0.099
avg_token_length,-0.115,-0.122,-0.131,-0.128,1.000,0.790,0.481,0.358,-0.012,0.376,-0.144,-0.115
avg_syllables_per_word,-0.065,-0.080,-0.071,-0.167,0.790,1.000,0.605,0.321,0.432,0.324,-0.149,-0.032
flesch_kincaid_grade,0.064,0.057,0.016,0.685,0.481,0.605,1.000,0.825,0.390,-0.093,0.004,0.056
gunning_fog_index,-0.024,-0.021,-0.074,0.728,0.358,0.321,0.825,1.000,0.169,-0.162,0.078,-0.075
lexical_density,0.061,0.049,0.059,0.087,-0.012,0.432,0.390,0.169,1.000,-0.084,0.032,0.094
lexical_diversity,-0.586,-0.593,-0.578,-0.412,0.376,0.324,-0.093,-0.162,-0.084,1.000,-0.134,-0.521


## 15. Final Summary And Observations

The final cell reports the processed document count, output paths, and key corpus averages.

In [27]:
def formatted_mean(series):
    value = series.mean(skipna=True)
    return 'n/a' if pd.isna(value) else f'{value:,.2f}'


print('Content feature extraction complete.')
print(f'Documents processed: {len(features_df):,}')
print(f'CSV output: {FEATURES_CSV_FILE}')
print(f'Parquet output: {FEATURES_PARQUET_FILE}' if features_parquet_written else 'Parquet output: skipped')
print(f'Config output: {CONFIG_FILE}')
print(f'Average word count: {formatted_mean(features_df["word_count"])}')
print(f'Average sentence count: {formatted_mean(features_df["sentence_count"])}')
print(f'Average Flesch-Kincaid grade: {formatted_mean(features_df["flesch_kincaid_grade"])}')
print(f'Read errors: {len(READ_ERRORS):,}')

Content feature extraction complete.
Documents processed: 21,058
CSV output: D:\University\FCSE\Courses\26S\TUG-26S-BT-Content_Optimization\data\content_features\content_features.csv
Parquet output: D:\University\FCSE\Courses\26S\TUG-26S-BT-Content_Optimization\data\content_features\content_features.parquet
Config output: D:\University\FCSE\Courses\26S\TUG-26S-BT-Content_Optimization\data\content_features\content_features_config.json
Average word count: 1,182.74
Average sentence count: 57.89
Average Flesch-Kincaid grade: 11.88
Read errors: 0
